In [79]:
import pathlib
import numpy as np
import datetime
import pyuvdata

In [ ]:
orig_freqs = np.array(
    [
        "27",
        "32",
        "36",
        "41",
        "46",
        "50",
        "55",
        "59",
        "64",
        "69",
        "73",
        "78",
        "82",
    ]
)
dates = ["2026-04-22"]

In [81]:
for date in dates:
    data_paths = []
    for freq in orig_freqs:
        data_paths.extend([str(p) for p in pathlib.Path(f"/lustre/pipeline/cosmology/{freq}MHz/{date}").rglob("*.ms") if p.is_dir()])

In [82]:
def populate_data_dict(data_paths):

    filenames = [path.split("/")[-1] for path in data_paths]
    times = np.array([
        datetime.datetime(
            int(filename[:4]),  # year
            int(filename[4:6]),  # month
            int(filename[6:8]),  # day
            int(filename[9:11]),  # hour
            int(filename[11:13]),  # minute
            int(filename[13:15]),  # second
        ) for filename in filenames
    ], dtype="datetime64")
    freqs = np.array([filename[16:18] for filename in filenames])
    data_dict = {
        "paths": data_paths,
        "freqs": freqs,
        "times": times,
    }
    return data_dict

In [115]:
def get_time_inds(data_dict, time_interval_minutes=2, time_integration_s=10):
    times = data_dict["times"]
    interval = np.timedelta64(int(time_interval_minutes), 'm')
    half_interval = interval / 2

    time_chunk_centers = np.arange(
        np.min(times) + (interval - np.timedelta64(int(time_integration_s), 's')) / 2,
        np.max(times) + half_interval,
        interval
    )

    time_inds = np.abs(times[:, None] - time_chunk_centers[None, :]).argmin(axis=1)
    data_dict["time_inds"] = time_inds
    return data_dict

In [116]:
def get_orig_freq_intervals():
    freq_interval_dict = {}
    freq_interval_dict["27"] = np.array([np.float64(27179687.5), np.float64(31749511.71875)])
    freq_interval_dict["32"] = np.array([np.float64(31773437.5), np.float64(36343261.71875)])
    freq_interval_dict["36"] = np.array([np.float64(36367187.5), np.float64(40937011.71875)])
    freq_interval_dict["41"] = np.array([np.float64(40960937.5), np.float64(45530761.71875)])
    freq_interval_dict["46"] = np.array([np.float64(45554687.5), np.float64(50124511.71875)])
    freq_interval_dict["50"] = np.array([np.float64(50148437.5), np.float64(54718261.71875)])
    freq_interval_dict["55"] = np.array([np.float64(54742187.5), np.float64(59312011.71875)])
    freq_interval_dict["59"] = np.array([np.float64(59335937.5), np.float64(63905761.71875)])
    freq_interval_dict["64"] = np.array([np.float64(63929687.5), np.float64(68499511.71875)])
    freq_interval_dict["69"] = np.array([np.float64(68523437.5), np.float64(73093261.71875)])
    freq_interval_dict["73"] = np.array([np.float64(73117187.5), np.float64(77687011.71875)])
    freq_interval_dict["78"] = np.array([np.float64(77710937.5), np.float64(82280761.71875)])
    freq_interval_dict["82"] = np.array([np.float64(82304687.5), np.float64(86874511.71875)])
    return freq_interval_dict

def get_new_freq_intervals():
    # Determined by where the equalization coefficients are held constant
    freq_interval_dict = {}
    freq_interval_dict["34"] = np.array([28352050.78125, 41295898.4375])
    freq_interval_dict["44"] = np.array([41319824.21875, 48354003.90625])
    freq_interval_dict["52"] = np.array([48377929.6875, 56823730.46875])
    freq_interval_dict["62"] = np.array([56847656.25, 67398925.78125])
    freq_interval_dict["72"] = np.array([67422851.5625, 77615234.375])
    freq_interval_dict["79"] = np.array([77639160.15625, 82256835.9375])
    freq_interval_dict["83"] = np.array([82280761.71875, 84649414.0625])
    return freq_interval_dict

In [117]:
def get_contributing_freqs(new_freq_interval_dict, orig_freq_interval_dict):
    contributing_freqs = []
    for new_freq in new_freq_interval_dict.keys():
        new_interval = new_freq_interval_dict[new_freq]
        contributing_freqs.append(
            [freq for freq, orig_interval in orig_freq_interval_dict.items() if orig_interval[0] <= new_interval[1] and orig_interval[1] >= new_interval[0]]
        )
    return contributing_freqs


In [118]:
new_freq_interval_dict = get_new_freq_intervals()
orig_freq_interval_dict = get_orig_freq_intervals()
contributing_freqs = get_contributing_freqs(new_freq_interval_dict, orig_freq_interval_dict)

In [119]:
print(contributing_freqs)

[['27', '32', '36', '41'], ['41', '46'], ['46', '50', '55'], ['55', '59', '64'], ['64', '69', '73'], ['73', '78'], ['78', '82']]


In [120]:
data_dict = populate_data_dict(data_paths)

In [121]:
data_dict = get_time_inds(data_dict)

In [122]:
print(data_dict["freqs"])

['27' '27' '27' ... '82' '82' '82']


In [123]:
for ind in np.where(data_dict["freqs"]=="82")[0]:
    print(data_dict["paths"][ind])

/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_080507_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_082741_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_081829_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_084043_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_081518_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_081458_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_083652_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_080006_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_083632_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_081609_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_082701_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_083843_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_080336_82MHz.ms
/lustre/pipeline/cosmology/82MHz/2026-04-22/08/20260422_080206_8

In [124]:
count = 0
for ind in np.where(data_dict["time_inds"]==0)[0]:
    print(data_dict["paths"][ind])
    count += 1
print(count)


/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054641_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054611_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054521_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054511_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054631_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054531_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054601_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054501_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054451_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054541_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054621_27MHz.ms
/lustre/pipeline/cosmology/27MHz/2026-04-22/05/20260422_054551_27MHz.ms
/lustre/pipeline/cosmology/32MHz/2026-04-22/05/20260422_054451_32MHz.ms
/lustre/pipeline/cosmology/32MHz/2026-04-22/05/20260422_054501_3

In [ ]:
def concatenate_files(data_dict, use_inds, output_freq, new_freq_interval_dict):

    uv_new = pyuvdata.UVData()
    filepaths = np.array([data_dict["paths"][ind] for ind in use_inds])
    frequencies = np.array([data_dict["freqs"][ind] for ind in use_inds])
    times = np.array([data_dict["times"][ind] for ind in use_inds])

    for time_ind, time in enumerate(np.unique(times)):  # Iterate over times
        for freq_ind, freq in enumerate(np.unique(frequencies)):  # Iterate over frequencies
            path_use = filepaths[np.where((times==time) & (frequencies==freq))]
            
            uv_new = pyuvdata.UVData()
            try:
                print(f"Reading file {path_use}")
                uv_new.read(path_use)
            except:
                print(f"WARNING: Error reading file {path_use}. Skipping.")
                return None

            # uv_new.select(polarizations=[-5, -6])
            uv_new.scan_number_array = None  # Added as a workaround for a pyuvdata bug (https://github.com/RadioAstronomySoftwareGroup/pyuvdata/issues/1595)
            if freq_ind == 0:
                uv_new.phase_to_time(np.min(uv_new.time_array))
                uv_single_freq = uv_new
            else:
                uv_new.phase_to_time(np.min(uv_single_freq.time_array))
                uv_single_freq.fast_concat(uv_new, "freq", inplace=True)
        if time_ind == 0:
            uv = uv_single_freq
        else:
            uv_single_freq.phase_to_time(np.min(uv.time_array))
            uv.fast_concat(uv_single_freq, "blt", inplace=True, run_check=False)
    


In [145]:
for time_ind in list(set(data_dict["time_inds"])):
    file_inds_time_match = np.where(data_dict["time_inds"]==time_ind)[0]
    for freq_ind, output_freq in enumerate(new_freq_interval_dict.keys()):
        file_inds_freq_match = np.where(np.array([freq in contributing_freqs[freq_ind] for freq in data_dict["freqs"]]))[0]
        use_inds = np.intersect1d(file_inds_time_match, file_inds_freq_match)
        if len(use_inds) != len(contributing_freqs[freq_ind]) * 12:
            print(f"ERROR: Not all files are present for time {time_ind} and frequency {output_freq}MHz.")
            continue
        else:
            print("Concatenating files")
            uv = concatenate_files(data_dict, use_inds, output_freq, new_freq_interval_dict)
            break

Concatenating files


Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_lo

Concatenating files


Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.
Setting telescope_location to value in known_telescopes for OVRO-LWA.


NameError: name 'path' is not defined